#### 重複的事件編號加上流水號

In [2]:
import pandas as pd

# 1. 讀取 CSV 檔案 (假設檔案名稱為 input.csv，且沒有標題列)
df = pd.read_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_bboxes_aoi_4326.csv')

# 2. 計算每個數值出現的總次數
counts = df['filename'].value_counts()

# 3. 定義轉換函數，用來追蹤目前是第幾次出現
occurrence_tracker = {}

def rename_duplicates(x):
    total_count = counts[x]
    
    # 如果總數大於 1，則加上後綴
    if total_count > 1:
        occurrence_tracker[x] = occurrence_tracker.get(x, 0) + 1
        return f"{x}_{occurrence_tracker[x]}"
    else:
        # 如果是唯一的，直接回傳字串
        return str(x)

# 4. 執行轉換
df['filename'] = df['filename'].apply(rename_duplicates)

# 5. 儲存結果到新的 CSV (不保留標題列與索引)
df.to_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_bboxes_aoi_4326_nameupdate.csv', index=False)

print("處理完成")

處理完成


#### 合併表格1

In [4]:
import pandas as pd

df_a = pd.read_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_bboxes_aoi_4326.csv')
df_b = pd.read_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_times_revised.csv')

# how='left' 會保留 A 檔案所有的列，並將 B 檔案對應的時間填入
df_combined = pd.merge(df_a, df_b, on='filename', how='left')

print("合併後的資料筆數：", len(df_combined))
print(df_combined.head())

# save
df_combined.to_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_bboxes_times_aoi.csv', index=False, encoding='utf-8-sig')

合併後的資料筆數： 329
   filename       minx       miny       maxx       maxy       west      south  \
0   1111002  14.815407  12.192891  14.905014  12.289329  14.815407  12.192891   
1   1111002  14.815407  12.192891  14.905014  12.289329  14.815407  12.192891   
2   1111002  14.815407  12.192891  14.905014  12.289329  14.815407  12.192891   
3   1111003  43.114643  11.506267  43.197556  11.626008  43.114643  11.506267   
4   1111003  43.114643  11.506267  43.197556  11.626008  43.114643  11.506267   

        east      north                      time1                      time2  
0  14.905014  12.289329  2020-06-23 04:46:21+00:00  2020-06-23 04:46:46+00:00  
1  14.905014  12.289329  2020-07-05 04:46:22+00:00  2020-07-05 04:46:47+00:00  
2  14.905014  12.289329  2020-09-15 04:46:26+00:00  2020-09-15 04:46:51+00:00  
3  43.197556  11.626008  2019-11-09 03:00:27+00:00  2019-11-09 03:00:52+00:00  
4  43.197556  11.626008  2019-11-15 02:59:38+00:00  2019-11-15 03:00:07+00:00  


#### 合併表格2

In [5]:
import pandas as pd

df_a = pd.read_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_bboxes_aoi_4326_nameupdate.csv')
df_b = pd.read_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_times_revised.csv')

# 1. 統一將 df_b 的 filename 轉為字串，避免型態衝突
df_b['filename'] = df_b['filename'].astype(str)

# 2. 在 df_a 建立一個暫時的 'join_key'，把 _1, _2 去掉 (例如 118_1 -> 118)
#    這樣才能跟 df_b 的原始檔名對起來
df_a['join_key'] = df_a['filename'].str.split('_').str[0]

# 3. 使用 join_key 與 df_b 的 filename 進行合併
df_combined = pd.merge(df_a, df_b, left_on='join_key', right_on='filename', how='left')

# 4. 合併後會多出一個重複的 filename 欄位 (來自 df_b)，可以視需求刪除
# df_combined = df_combined.drop(columns=['filename_y', 'join_key']).rename(columns={'filename_x': 'filename'})
# 或者簡單處理：
# df_combined = df_combined.drop(columns=['join_key'])

print("合併後的資料筆數：", len(df_combined))
df_combined.to_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_bboxes_times_aoi.csv', index=False, encoding='utf-8-sig')

合併後的資料筆數： 329


#### 讀取CSV檔案並查看其bounding box和最早及最晚時間

In [3]:
# 讀取CSV檔案並查看其bounding box和最早及最晚時間
import pandas as pd
csv_path = "/home/chunen/nas/bigdata/kurosiwo/kurosiwo_bboxes_times.csv"
df = pd.read_csv(csv_path)

# 查看最早和最晚時間
print("\nTime Range:")
print(f"Earliest Time: {df['time1'].min()}")
print(f"Latest Time: {df['time2'].max()}")

# 查看bounding box的範圍
print("\nBounding Box Range:")
print(f"Min west: {df['west'].min()}")
print(f"Max west: {df['west'].max()}")
print(f"Min east: {df['east'].min()}")
print(f"Max east: {df['east'].max()}")
print(f"Min south: {df['south'].min()}")
print(f"Max south: {df['south'].max()}")
print(f"Min north: {df['north'].min()}")
print(f"Max north: {df['north'].max()}")


Time Range:
Earliest Time: 2014-10-17 18:11:12+00:00
Latest Time: 2022-09-11 01:26:41+00:00

Bounding Box Range:
Min west: -97.2682764072998
Max west: 149.1696995887029
Min east: -95.53776184397196
Max east: 153.3752524228369
Min south: -33.811469560127314
Max south: 57.41425786480876
Min north: -29.110792427805563
Max north: 59.31353662378383


#### 將子資料夾內特定副檔名（如 .tif）的檔案統整到根目錄

In [1]:
from pathlib import Path
import shutil

def gather_tif_files(target_folder_path):
    # 將路徑字串轉換為 Path 物件
    folder_A = Path(target_folder_path)
    
    # 確保資料夾存在
    if not folder_A.exists() or not folder_A.is_dir():
        print(f"錯誤：找不到資料夾 {target_folder_path}")
        return

    # 使用 rglob 找出 A 資料夾下（包含所有子資料夾）所有的 .tif 檔案
    # 支援大小寫副檔名，例如 .TIF 或 .tif，建議可以轉小寫比對或直接搜尋
    tif_files = list(folder_A.rglob('*.tif')) + list(folder_A.rglob('*.TIF'))
    
    moved_count = 0

    for file_path in tif_files:
        # 檢查該檔案是否「已經」在 A 資料夾的根目錄下
        # 如果已經在根目錄，就不需要移動
        if file_path.parent != folder_A:
            # 設定目標路徑（A 資料夾根目錄 + 原本的檔名）
            target_path = folder_A / file_path.name
            
            # 【防呆機制】如果 A 資料夾已經有同名檔案，則在檔名後加上數字
            counter = 1
            while target_path.exists():
                # file_path.stem 是主檔名，file_path.suffix 是副檔名 (.tif)
                new_name = f"{file_path.stem}_{counter}{file_path.suffix}"
                target_path = folder_A / new_name
                counter += 1
            
            # 移動檔案
            shutil.move(str(file_path), str(target_path))
            print(f"已移動: {file_path.name} -> {target_path.name}")
            moved_count += 1

    print(f"\n處理完成！共移動了 {moved_count} 個 .tif 檔案。")

# 執行區塊（請將下方路徑替換成您實際的 A 資料夾路徑）
# Windows 路徑範例: r"C:\Users\Username\Desktop\Folder_A"
# Mac/Linux 路徑範例: "/Users/Username/Desktop/Folder_A"
if __name__ == "__main__":
    YOUR_FOLDER_PATH = r"/home/chunen/nas/bigdata/final/S2_modify" 
    gather_tif_files(YOUR_FOLDER_PATH)

已移動: 256_deflate_174_20150814T094910-0000023296-0000023296.tif -> 256_deflate_174_20150814T094910-0000023296-0000023296.tif
已移動: 256_deflate_174_20160805T093918-0000023296-0000023296.tif -> 256_deflate_174_20160805T093918-0000023296-0000023296.tif
已移動: 256_deflate_174_20150814T094910-0000023296-0000000000.tif -> 256_deflate_174_20150814T094910-0000023296-0000000000.tif
已移動: 256_deflate_174_20160805T093918-0000023296-0000000000.tif -> 256_deflate_174_20160805T093918-0000023296-0000000000.tif
已移動: 512_deflate_174_20150814T094910-0000023296-0000023296.tif -> 512_deflate_174_20150814T094910-0000023296-0000023296.tif
已移動: 512_deflate_174_20160805T093918-0000023296-0000023296.tif -> 512_deflate_174_20160805T093918-0000023296-0000023296.tif
已移動: 256_deflate_174_20150814T094910-0000000000-0000023296.tif -> 256_deflate_174_20150814T094910-0000000000-0000023296.tif
已移動: 512_deflate_174_20150814T094910-0000023296-0000000000.tif -> 512_deflate_174_20150814T094910-0000023296-0000000000.tif
已移動: 512

#### 將子資料夾內特定副檔名（如 .tif）的檔案依檔名前綴統整到個別資料夾

In [1]:
from pathlib import Path
import shutil

def organize_files_by_prefix(source_folder_path, destination_base_path):
    # 將路徑轉換為 Path 物件
    source_folder = Path(source_folder_path)
    dest_base_folder = Path(destination_base_path)
    
    # 確保來源資料夾存在
    if not source_folder.exists() or not source_folder.is_dir():
        print(f"錯誤：找不到來源資料夾 {source_folder_path}")
        return

    # 找出所有的 .tif 與 .TIF 檔案
    tif_files = list(source_folder.rglob('*.tif')) + list(source_folder.rglob('*.TIF'))
    
    moved_count = 0
    skipped_count = 0

    for file_path in tif_files:
        filename = file_path.name
        
        # 【關鍵步驟 1】檢查檔名中是否有底線 "_"
        if "_" not in filename:
            print(f"略過: {filename} (沒有底線，無法判斷分類數字)")
            skipped_count += 1
            continue
            
        # 【關鍵步驟 2】取得底線前面的數字字串
        # 例如 "174_20150814.tif".split("_") 會變成 ["174", "20150814.tif"]
        # 我們取索引值 [0] 也就是 "174"
        prefix = filename.split("_")[0]
        
        # 【關鍵步驟 3】設定該數字專屬的目標資料夾路徑 (例如: 集中資料夾/174)
        dest_folder = dest_base_folder / prefix
        
        # 自動建立該數字的資料夾 (如果已經存在就不會報錯)
        dest_folder.mkdir(parents=True, exist_ok=True)

        # 【安全檢查】如果檔案已經在正確的分類資料夾裡，就跳過
        if file_path.parent == dest_folder:
            continue

        target_path = dest_folder / filename
        
        # 【防呆機制】處理同名檔案衝突
        counter = 1
        while target_path.exists():
            new_name = f"{file_path.stem}_{counter}{file_path.suffix}"
            target_path = dest_folder / new_name
            counter += 1
        
        # 移動檔案
        shutil.move(str(file_path), str(target_path))
        print(f"已分類: {filename} -> 【{prefix}】資料夾")
        moved_count += 1

    print(f"\n自動分類完成！")
    print(f"成功分類並移動了 {moved_count} 個檔案。")
    if skipped_count > 0:
        print(f"有 {skipped_count} 個檔案因為沒有底線而被略過。")

# ================= 執行區塊 =================
if __name__ == "__main__":
    # 1. 來源路徑：存放雜亂 .tif 檔案的根目錄
    SOURCE_PATH = r"/home/chunen/nas/bigdata/final/S2_aoi" 
    
    # 2. 輸出路徑：您希望在哪裡產生這些「數字資料夾」
    # (您也可以設為同一個 A資料夾，這樣就會直接在 A資料夾 內建立 174、267 等資料夾)
    DESTINATION_BASE = r"/home/chunen/nas/bigdata/final/S2_aoi_modify" 
    
    # 執行程式
    organize_files_by_prefix(SOURCE_PATH, DESTINATION_BASE)

已分類: 502_6_20210224T114659.tif -> 【502】資料夾
已分類: 502_6_20210212T115655.tif -> 【502】資料夾
已分類: 502_6_20210130T114700.tif -> 【502】資料夾
已分類: 502_5_20210224T114659.tif -> 【502】資料夾
已分類: 502_5_20210212T115655.tif -> 【502】資料夾
已分類: 502_5_20210130T114704.tif -> 【502】資料夾
已分類: 502_4_20210224T114702.tif -> 【502】資料夾
已分類: 502_4_20210212T115700.tif -> 【502】資料夾
已分類: 502_4_20210212T115700 - 2026-04-18T20:53:05.756Z.tif -> 【502】資料夾
已分類: 502_4_20210130T114704.tif -> 【502】資料夾
已分類: 502_4_20210130T114704 - 2026-04-18T20:50:55.744Z.tif -> 【502】資料夾
已分類: 502_3_20210224T114659.tif -> 【502】資料夾
已分類: 502_3_20210224T114659 - 2026-04-18T20:48:39.893Z.tif -> 【502】資料夾
已分類: 502_3_20210212T115655.tif -> 【502】資料夾
已分類: 502_3_20210130T114700.tif -> 【502】資料夾
已分類: 502_3_20210130T114700 - 2026-04-18T20:42:03.017Z.tif -> 【502】資料夾
已分類: 502_2_20210224T114659.tif -> 【502】資料夾
已分類: 502_2_20210212T115655.tif -> 【502】資料夾
已分類: 502_2_20210212T115655 - 2026-04-18T20:36:01.423Z.tif -> 【502】資料夾
已分類: 502_2_20210130T114700.tif -> 【502】資料夾
已分類: 

#### 檢查哪些影像是全黑、哪些影像是正常

In [1]:
import os
import glob
import math
import rasterio
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# 1️⃣ 設定你的目標根目錄以及輸出位置
base_dir = "/home/chunen/nas/bigdata/final/S2_aoi_modify"
output_csv = "/home/chunen/nas/bigdata/final/S2_aoi_modify/S2_image_status_report.csv"

# 準備儲存結果的列表
results = []

# 2️⃣ 使用 glob 找出所有子資料夾中的 .tif 檔案
print("🔍 正在搜尋檔案...")
tif_files = glob.glob(os.path.join(base_dir, "**", "*.tif"), recursive=True)
print(f"找到共 {len(tif_files)} 個 TIFF 檔案，開始分析影像內容...\n")

for file_path in tqdm(tif_files, desc="影像分析進度"):
    try:
        with rasterio.open(file_path) as src:
            # 為了加快速度，我們只讀取第一個波段來計算比例即可
            data = src.read(1)
            total_pixels = data.size
            
            # 3️⃣ 判斷哪些是有效像素
            nodata_val = src.nodata
            if nodata_val is not None:
                if math.isnan(nodata_val):
                    valid_mask = ~np.isnan(data)
                else:
                    valid_mask = (data != nodata_val)
            else:
                # 如果 GEE 沒有寫入 nodata 屬性，通常空值會被填為 0
                valid_mask = (data != 0)
                
            valid_pixels = np.sum(valid_mask)
            valid_ratio = valid_pixels / total_pixels
            
            # 4️⃣ 分類邏輯
            if valid_ratio == 0:
                status = "1. No data at all"
            elif valid_ratio == 1.0:
                status = "3. Normal"
            else:
                # 顯示百分比方便你判斷 (例如 99.9% 有效可能只是邊界稍微切到)
                status = "2. Partial normal"
                
            results.append({
                "Folder": os.path.basename(os.path.dirname(file_path)),
                "Filename": os.path.basename(file_path),
                "Status": status,
                "Valid_Percent (%)": round(valid_ratio * 100, 2)
            })
            
    except Exception as e:
        tqdm.write(f"❌ 讀取 {os.path.basename(file_path)} 失敗: {e}")
        results.append({
            "Folder": os.path.basename(os.path.dirname(file_path)),
            "Filename": os.path.basename(file_path),
            "Status": "讀取失敗",
            "Valid_Percent (%)": -1
        })

# 5️⃣ 轉換成 DataFrame 並輸出報告
df = pd.DataFrame(results)

print("📊 【影像狀態統計報表】")
print(df["Status"].value_counts().to_string())

# 將詳細清單存成 CSV，方便你後續開啟 Excel 篩選與刪除檔案
df.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"\n💾 詳細清單已儲存至：{output_csv}")

# 如果你想直接在 Jupyter 看「全黑」的檔案清單
black_images = df[df["Status"] == "1. 全黑"]
if not black_images.empty:
    print(f"\n⚠️ 發現 {len(black_images)} 張全黑影像，前 5 筆如下：")
    print(black_images.head().to_string(index=False))

🔍 正在搜尋檔案...
找到共 236 個 TIFF 檔案，開始分析影像內容...



影像分析進度:   0%|          | 0/236 [00:00<?, ?it/s]

❌ 讀取 1111009_1_20220607T061118-0000000000-0000000000.tif 失敗: Read failed. See previous exception for details.
📊 【影像狀態統計報表】
Status
3. Normal            180
2. Partial normal     53
1. No data at all      2
讀取失敗                   1

💾 詳細清單已儲存至：/home/chunen/nas/bigdata/final/S2_aoi_modify/S2_image_status_report.csv


#### 壓縮多個資料夾

In [2]:
import os
import tarfile
import zipfile
from tqdm import tqdm
from pathlib import Path

# 設定
source_root = Path("/mnt/hdd/KuroSiwo_data/KuroSiwo/data")
output_dir = Path("/home/chunen/nas/bigdata/final/kurosiwo_S1_DEM") # 指定儲存位置

# 確保輸出目錄存在
output_dir.mkdir(parents=True, exist_ok=True)

# 篩選出所有子資料夾（排除隱藏資料夾）
all_dirs = [d for d in source_root.iterdir() if d.is_dir() and not d.name.startswith('.')]


# print(f"開始壓縮，目標格式為 .tar.gz...")

# # 顯示進度條
# for folder in tqdm(all_dirs, desc="總體進度"):
#     target_file = output_dir / f"{folder.name}.tar.gz"
    
#     with tarfile.open(target_file, "w:gz") as tar:
#         # arcname=folder.name 確保壓縮檔解壓後不會包含冗長的層級路徑
#         tar.add(folder, arcname=folder.name)

print(f"開始壓縮，目標格式為 zip...")

# 總進度條：顯示處理了多少個資料夾
for folder in tqdm(all_dirs, desc="總體壓縮進度"):
    # 設定輸出的完整路徑 (例如: /指定路徑/資料夾名.zip)
    zip_file_path = output_dir / f"{folder.name}.zip"
    
    # 建立 ZIP 檔
    with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # 遍歷資料夾內的所有檔案與子目錄
        for root, dirs, files in os.walk(folder):
            for file in files:
                # 建立檔案的絕對路徑
                file_path = os.path.join(root, file)
                # 計算在壓縮檔內的相對路徑，避免解壓後出現冗長的層級
                arcname = os.path.relpath(file_path, folder.parent)
                zipf.write(file_path, arcname)

print(f"\n全部壓縮完成！檔案儲存在: {output_dir}")

開始壓縮，目標格式為 zip...


總體壓縮進度:   0%|          | 0/43 [00:00<?, ?it/s]

總體壓縮進度:   0%|          | 0/43 [04:33<?, ?it/s]


KeyboardInterrupt: 

#### 根據bbox來新增洲別欄位

In [5]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. 讀取你的 CSV 檔案
df = pd.read_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_final.csv', encoding='cp950')

# 2. 計算中心點並建立 Geometry 欄位
# 假設你的座標是 WGS84 (經緯度)
df['center_lon'] = (df['west'] + df['east']) / 2
df['center_lat'] = (df['south'] + df['north']) / 2

# 將 DataFrame 轉換為 GeoDataFrame
geometry = [Point(xy) for xy in zip(df['center_lon'], df['center_lat'])]
gdf_events = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# 3. 讀取包含「大洲」資訊的地理資料 (使用 Natural Earth 的 GitHub 鏡像)
# 這個檔案包含 'CONTINENT' 欄位
world_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
world = gpd.read_file(world_url)

# 【關鍵步驟】檢查欄位名稱，避免 KeyError
# Natural Earth 數據通常是大寫的 'CONTINENT'
continent_col = 'CONTINENT' if 'CONTINENT' in world.columns else 'continent'

if continent_col not in world.columns:
    print("警告：此地圖資料不包含大洲資訊，可用欄位有：", world.columns)
else:
    # 4. 進行空間連接 (Spatial Join)
    # 只選取大洲和幾何圖形欄位
    result = gpd.sjoin(gdf_events, world[[continent_col, 'geometry']], how="left", predicate='within')
    
    # 將欄位名稱統一改回小寫 'continent' 方便後續使用
    result = result.rename(columns={continent_col: 'continent'})

    # 5. 輸出結果
    print(result[['continent', 'geometry']].head())
    result.to_csv('/home/chunen/nas/bigdata/final/kurosiwo/kurosiwo_final_plus.csv', index=False)

  continent                   geometry
0    Europe  POINT (-2.53913 42.14628)
1      Asia  POINT (95.16451 17.02629)
2    Europe  POINT (-1.71619 54.19901)
3    Europe  POINT (20.67711 41.74447)
4      Asia   POINT (80.66229 6.47908)
